# Finding a Yiddish corpus in 947 GB — without downloading 947 GB

### A gentle introduction to Institutional Books, DuckDB, and targeted text retrieval

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PULdischo/discho-blog/blob/main/src/assets/notebooks/institutional-books-yiddish.ipynb)

[Institutional Books 1.0](https://huggingface.co/datasets/institutional/institutional-books-hl)
is 983,004 public-domain books digitised from Harvard Library — about **947 GB** of
text spread across thousands of files. That number is intimidating, but almost no
research question needs all of it. This post walks, slowly, through pulling a single
language subset — every Yiddish book in the collection — without downloading the rest.

By the end you will know how to:

1. **filter** the collection using a small, open metadata file;
2. **estimate** how large your chosen subset is before fetching anything;
3. **fetch** only the books you want; and
4. **save** them for your own work.

A few terms up front — every one of them appears again below:

| term | what it means |
|---|---|
| **Parquet** | a columnar file format; you can read one column without reading the rest |
| **metadata** | catalogue fields — title, author, year, language — in one small file |
| **full text** | the actual OCR text of a book, in the large 947 GB dataset |
| **OCR** | text a machine read from scanned page images (imperfect for old books) |
| **DuckDB** | a local SQL engine that queries Parquet files directly |
| **barcode_src** | the identifier that links a metadata row to its full text |
| **shards** | the many Parquet files the full text is split across |
| **manifest** | a reproducible list of *which* books you selected — not the text itself |

> **Licence.** The metadata is open. The full-text dataset is released under the
> [IDI Terms of Use](https://institutionaldatainitiative.org/) — **noncommercial use
> only, and no redistribution of the text.** So we keep the text for our own use and,
> if we want to share, publish a manifest of identifiers rather than the books.

## 1. Setup

`duckdb` does the heavy lifting. Its `httpfs` extension can read a Parquet file over
HTTPS with *range requests*: it fetches just the file's footer, works out which parts
of the file your query actually needs, and downloads only those bytes. That is what
lets us touch a 947 GB dataset without downloading it.

In [1]:
!pip -q install "duckdb>=1.1.0" huggingface_hub datasets pandas matplotlib


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import duckdb, threading, time
import pandas as pd
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor, as_completed

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql("SET enable_progress_bar = false;")   # avoid blank progress bars in the notebook
con.sql("SELECT version() AS duckdb_version").show()

┌────────────────┐
│ duckdb_version │
│    varchar     │
├────────────────┤
│ v1.5.5         │
└────────────────┘



## 2. Get the metadata once, then query it locally

The metadata is a **single ~306 MB Parquet file**, and it is **not gated** — no login
or token needed. We could point DuckDB straight at its `hf://` URL, but every query
would hit the network, and the Hub rate-limits anonymous requests (HTTP 429) once a
notebook fires enough of them. So we download it **once**, cache it, and query the
local copy. Every question about *which* books we want is answered from this one file.

In [5]:
from huggingface_hub import hf_hub_download

META = hf_hub_download(
    "institutional/institutional-books-hl-metadata",
    "data/train-00000-of-00001.parquet",
    repo_type="dataset",
)
print(META)

# The schema. Note that the book text is NOT here - only in the 947 GB dataset.
con.sql(f"DESCRIBE SELECT * FROM '{META}'").df()

/home/apjanco/.cache/huggingface/hub/datasets--institutional--institutional-books-hl-metadata/snapshots/e0c0b860482e49cced246c5cd7c0d22ac8a86040/data/train-00000-of-00001.parquet


,column_name,column_type,null,key,default,extra
0,barcode_src,VARCHAR,YES,None,None,None
1,title_src,VARCHAR,YES,None,None,None
2,author_src,VARCHAR,YES,None,None,None
3,date1_src,VARCHAR,YES,None,None,None
4,date2_src,VARCHAR,YES,None,None,None
5,date_types_src,VARCHAR,YES,None,None,None
6,page_count_src,INTEGER,YES,None,None,None
7,token_count_o200k_base_gen,INTEGER,YES,None,None,None
8,language_src,VARCHAR,YES,None,None,None
9,language_gen,VARCHAR,YES,None,None,None


In [4]:
con.sql(f"SELECT count(*) AS n_books FROM '{META}'").show()

con.sql(f"""
    SELECT barcode_src, title_src, author_src, date1_src,
           language_src, language_gen
    FROM '{META}'
    LIMIT 5
""").show()

┌─────────┐
│ n_books │
│  int64  │
├─────────┤
│  983004 │
└─────────┘

┌────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────┬───────────┬──────────────┬──────────────┐
│  barcode_src   │                                                                            title_src                                                                             │        author_src         │ date1_src │ language_src │ language_gen │
│    varchar     │                                                                             varchar                                                                              │          varchar          │  varchar  │   varchar    │   varchar    │
├────────────────┼─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

That ran against the local file — instant, and with zero network requests. The columns
we care about for this post:

| column | what it is |
|---|---|
| `barcode_src` | the join key to the full-text dataset |
| `title_src`, `author_src` | from the catalogue record |
| `language_src` | language in the catalogue record (ISO 639-3) |
| `language_gen` | language **detected from the OCR text** (ISO 639-3) |
| `date1_src` | publication year as a string — messy (`"18uu"`, `""`, ranges) |
| `token_count_o200k_base_gen` | size of the book in tokens — handy for size estimates |
| `ocr_score_src` | OCR quality (higher is better) |

Notice there are **two** language columns. `language_src` is what a human cataloguer
recorded, possibly decades ago; `language_gen` is what a language detector saw in the
OCR text. They do not always agree — which is exactly why we look before we filter.

## 3. Find the right language code

Yiddish is not a single tidy label. It has more than one ISO 639-3 code — `yid`
(the macrolanguage), `ydd` (Eastern Yiddish), `yih` (Western Yiddish) — and detectors
don't always pick the same one. Rather than assume, let's count what actually appears
in each language column.

In [6]:
con.sql(f"""
    SELECT 'gen' AS field, language_gen AS code, count(*) AS n
    FROM '{META}' WHERE language_gen LIKE 'y%' GROUP BY language_gen
    UNION ALL
    SELECT 'src', language_src, count(*)
    FROM '{META}' WHERE language_src LIKE 'y%' GROUP BY language_src
    ORDER BY n DESC
""").show()

┌─────────┬─────────┬───────┐
│  field  │  code   │   n   │
│ varchar │ varchar │ int64 │
├─────────┼─────────┼───────┤
│ gen     │ ydd     │   826 │
│ src     │ yid     │   687 │
│ gen     │ yad     │     4 │
│ gen     │ yao     │     4 │
│ gen     │ yua     │     3 │
│ src     │ yor     │     3 │
│ gen     │ yrk     │     1 │
│ gen     │ ykg     │     1 │
│ src     │ yao     │     1 │
│ src     │ yap     │     1 │
└─────────┴─────────┴───────┘
  10 rows         3 columns



In [7]:
YIDDISH_CODES = ("yid", "ydd")   # `yih` (Western Yiddish) shows ~0 books above, so drop it

yiddish = con.sql(f"""
    SELECT barcode_src, title_src, author_src, date1_src,
           page_count_src, token_count_o200k_base_gen,
           language_src, language_gen, ocr_score_src
    FROM '{META}'
    WHERE language_gen IN {YIDDISH_CODES}
    ORDER BY token_count_o200k_base_gen DESC NULLS LAST
""").df()

print(f"{len(yiddish):,} books")
yiddish.head(10)

826 books


,barcode_src,title_src,author_src,date1_src,page_count_src,token_count_o200k_base_gen,language_src,language_gen,ocr_score_src
0,HWNM7F,Sidur Ḳorban minḥah : nusaḥ Sefarad [kolel kol...,,1897,1076,2881147,heb,ydd,80
1,HN9GUH,Tseʾenah u-reʾenah : ʻal Ḥamishah Ḥumshe Torah.,"Ashkenazi, Yaakov ben Yitzchak",1895,810,2273639,yid,ydd,75
2,HWMCS7,Sidur Ḳorban minḥah : kolel kol tefilot ha-sha...,,1892,958,2236153,heb,ydd,79
3,HWMJGT,Menorat ha-maʼor : ʻim perush ... Nefesh Yehud...,"Aboab, Isaac",1861,488,2199857,heb,ydd,81
4,HWNMXX,Miḳraʼot gedolot : ...min ḥamishah humshe Tora...,,1902,708,2142871,heb,ydd,64
5,HN9GQ8,Nofet tsufim : ʻal Ḥamishah Ḥumshe Torah ṿe-ḥa...,Zevi Hirsch ben Yeraḥmiel.,1882,648,2015552,yid,ydd,78
6,HWMKCL,[Ḥovot ha-levavot] : Ḥovot ha-levavot,Baḥya ben Joseph ibn Paḳuda,1866,376,1995435,heb,ydd,66
7,HWMTPN,Menorat ha-maʼor : ... ʻim perush Nefesh Yehud...,"Aboab, Isaac",1875,374,1892182,heb,ydd,81
8,HN9LB9,"Miḳraʼot gedolot : ḥamishah ḥumshe Torah, ʻim ...",,1888,536,1705072,heb,ydd,62
9,HWMJHD,Menorat ha-maʼor : ʻim perush Nefesh Yehudah ...,"Aboab, Isaac",1878,336,1688254,heb,ydd,81


## 4. How big is "all of it", really?

Before fetching anything, estimate the download. Across the whole collection the text
averages roughly **1 MB per book** (947 GB / 983k books). Multiply that by the number
of Yiddish books and you have a rough size — enough to confirm the subset is small
enough to just take all of it.

In [8]:
AVG_MB_PER_BOOK = 947_000 / 983_004     # ~0.96 MB

est_gb = len(yiddish) * AVG_MB_PER_BOOK / 1000
toks = yiddish.token_count_o200k_base_gen.sum()
print(f"Yiddish: {len(yiddish):,} books  ~{est_gb:.2f} GB  {toks/1e6:,.0f}M tokens")

# For comparison, a few other small language communities:
con.sql(f"""
    SELECT language_gen AS code, count(*) AS n_books,
           round(count(*) * {AVG_MB_PER_BOOK} / 1000, 2) AS est_gb,
           round(sum(token_count_o200k_base_gen) / 1e6, 1) AS m_tokens
    FROM '{META}'
    WHERE language_gen IN ('yid','ydd','yih','haw','cym','gla','gle')
    GROUP BY language_gen
    ORDER BY n_books DESC
""").show()

Yiddish: 826 books  ~0.80 GB  194M tokens
┌─────────┬─────────┬────────┬──────────┐
│  code   │ n_books │ est_gb │ m_tokens │
│ varchar │  int64  │ double │  double  │
├─────────┼─────────┼────────┼──────────┤
│ cym     │    1742 │   1.68 │    608.7 │
│ ydd     │     826 │    0.8 │    193.6 │
│ gle     │     234 │   0.23 │     26.9 │
│ gla     │     162 │   0.16 │     28.3 │
│ haw     │      33 │   0.03 │      8.6 │
└─────────┴─────────┴────────┴──────────┘



Whatever the exact number, it is a *tiny* fraction of 947 GB. Downloading the whole
Yiddish subset is completely reasonable; downloading the whole dataset to get it is not.
That gap — a few hundred MB versus 947 GB — is the entire point of filtering first.

> **A caveat about OCR quality.** A cleaned, post-processed text field exists only for
> the five largest languages (`eng`, `deu`, `fra`, `ita`, `spa`). For Yiddish you get
> the raw OCR, and Yiddish is printed in Hebrew script, which older OCR handles poorly.
> So check `ocr_score_src` before you rely on the text.

In [9]:
yiddish.ocr_score_src.describe()

count    826.000000
mean      81.027845
std       13.476730
min        0.000000
25%       75.000000
50%       83.000000
75%       91.000000
max       96.000000
Name: ocr_score_src, dtype: float64

## 5. Authenticate, then fetch the text

The full-text dataset is **gated** (access is auto-approved, but you must accept the
terms). One-time steps:

1. Log in to Hugging Face and open
   [institutional/institutional-books-hl](https://huggingface.co/datasets/institutional/institutional-books-hl).
2. Accept the IDI early-access terms (instant).
3. Create a **read** token.

Then authenticate this session:

In [10]:
from huggingface_hub import notebook_login
notebook_login()

In [11]:
# Let DuckDB pick up the cached HF token for hf:// range requests.
con.sql("DROP SECRET IF EXISTS hf;")
con.sql("CREATE SECRET hf (TYPE huggingface, PROVIDER credential_chain);")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

### Fetching without sweeping all 9,831 shards

The full text is split across thousands of Parquet **shards** named
`train-00000-of-09831.parquet`, `train-00001-...`, and so on. The naive way to collect
our books is to run a `WHERE barcode_src IN (...)` query against *every* shard — correct,
but slow, because it touches all 9,831 files.

There is no column in the metadata that says "book X is in shard Y". But we don't need
one: the metadata rows are in the **same physical order** as the shards laid end to end.
So if each shard holds `rows_per_shard` books, the book at metadata row `i` lives in
shard `i // rows_per_shard`. We verify that assumption once against shard 0, then compute
the exact set of shards our barcodes fall in and query only those. For a tight subset
like the Yiddish books that turns a 9,831-file sweep into a handful of files.

The fetch still adds three practical touches so it stays reliable:

* **Concurrency.** The bottleneck is per-file network latency, not bandwidth, so we
  query shards from a thread pool — one DuckDB connection per worker.
* **Retries.** Each shard retries a few times with backoff, so a stray 429 or dropped
  connection doesn't kill the whole run.
* **Early exit.** The moment every requested barcode has been found, we stop.

You don't need to follow every line — call it like a black box: give it a list of
barcodes, get back a DataFrame of text.

In [12]:
# Do the shards line up with the metadata row order? Download shard 0 once and compare
# its barcodes to the first N rows of the metadata, in order. (Shard 0 is ~100 MB - a
# one-time cost that pays for itself the moment you skip the other 9,830 shards.)
HL_REPO  = "institutional/institutional-books-hl"
N_SHARDS = 9831
HL_URL   = f"hf://datasets/{HL_REPO}/data/train-{{:05d}}-of-0{N_SHARDS}.parquet"


def shard_name(i):
    return f"data/train-{i:05d}-of-0{N_SHARDS}.parquet"


con.sql("SET threads TO 1;")   # so DuckDB reads rows in the file's physical order

shard0 = hf_hub_download(HL_REPO, shard_name(0), repo_type="dataset")
s0 = con.sql(f"SELECT barcode_src FROM read_parquet('{shard0}')").df().barcode_src.tolist()
ROWS_PER_SHARD = len(s0)

meta_head = con.sql(
    f"SELECT barcode_src FROM '{META}' LIMIT {ROWS_PER_SHARD}"
).df().barcode_src.tolist()

aligned = s0 == meta_head
print("rows per shard:", ROWS_PER_SHARD)
print("shard 0 == metadata rows 0..N, in order:", aligned)


def _shards_for(barcodes):
    """Which shard(s) hold these barcodes, from the metadata row order.

    If the alignment check failed, fall back to sweeping every shard - slower, but
    always correct.
    """
    if not aligned:
        return list(range(N_SHARDS))
    con.execute("SET threads TO 1;")   # row_number() must follow physical order
    bc = "', '".join(barcodes)
    return con.sql(f"""
        WITH numbered AS (
            SELECT barcode_src, row_number() OVER () - 1 AS ridx FROM '{META}'
        )
        SELECT DISTINCT (ridx // {ROWS_PER_SHARD})::INT AS shard
        FROM numbered WHERE barcode_src IN ('{bc}')
        ORDER BY shard
    """).df().shard.tolist()

rows per shard: 100
shard 0 == metadata rows 0..N, in order: True


In [13]:
_tl = threading.local()


def _worker_con():
    """One DuckDB connection per worker thread, each with its own HF secret."""
    if not hasattr(_tl, "con"):
        c = duckdb.connect()
        c.execute("INSTALL httpfs; LOAD httpfs;")
        c.execute("CREATE SECRET hf (TYPE huggingface, PROVIDER credential_chain);")
        c.execute("SET threads TO 1;")
        c.execute("SET enable_progress_bar = false;")
        _tl.con = c
    return _tl.con


def fetch_text(barcodes, concurrency=16, retries=4):
    """Fetch full text for a list of barcodes, touching only the shards they live in."""
    barcodes = list(barcodes)
    want = set(barcodes)
    bc_sql = "', '".join(barcodes)
    shards = _shards_for(barcodes)
    print(f"{len(barcodes)} books -> {len(shards)} shard(s) "
          f"{'(exact set)' if aligned else '(full sweep)'}", flush=True)

    def one(idx):
        q = f"""
            SELECT barcode_src, title_src, language_gen,
                   array_to_string(text_by_page_src, chr(10)) AS text_src
            FROM read_parquet('{HL_URL.format(idx)}')
            WHERE barcode_src IN ('{bc_sql}')
        """
        for attempt in range(retries):
            try:
                return _worker_con().execute(q).fetchdf()
            except Exception as e:
                if attempt == retries - 1:
                    print(f"  shard {idx} gave up: {e}", flush=True)
                    return None
                time.sleep(3 * (attempt + 1))

    out, found, done = [], set(), 0
    with ThreadPoolExecutor(max_workers=concurrency) as ex:
        futs = {ex.submit(one, i): i for i in shards}
        for f in as_completed(futs):
            done += 1
            r = f.result()
            if r is not None and len(r):
                out.append(r)
                found |= set(r.barcode_src)
            if done % 200 == 0 or found == want:
                print(f"  {done}/{len(shards)} shards | {len(found)}/{len(want)} books",
                      flush=True)
            if found == want:                       # every barcode accounted for
                for fut in futs:
                    fut.cancel()
                break

    cols = ["barcode_src", "title_src", "language_gen", "text_src"]
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame(columns=cols)

In [14]:
yiddish_text = fetch_text(yiddish.barcode_src.tolist())

got_mb = yiddish_text.text_src.str.len().sum() / 1e6
print(f"Fetched {len(yiddish_text)} / {len(yiddish)} books, "
      f"~{got_mb:.0f} MB of text  (vs 947,000 MB for the whole dataset)")
yiddish_text[["barcode_src", "title_src", "language_gen"]].head()

826 books -> 144 shard(s) (exact set)
  144/144 shards | 826/826 books
Fetched 826 / 826 books, ~387 MB of text  (vs 947,000 MB for the whole dataset)


,barcode_src,title_src,language_gen
0,32044087831202,Hilkhot sheḥiṭah u-vediḳah : in Dayṭsher shpra...,ydd
1,32044087831210,Amirah le-vet Yaʻaḳov : anshprakhe an das hoys...,ydd
2,32044089108120,"Yiddishe in Hebrőüshe Lieder, Folks in naziona...",ydd
3,32044072263296,Die Trigonaloiden des Zoologischen Museums der...,ydd
4,32044088300793,Di Idishe gishikhṭe : fun Avrom Ovinu biz di y...,ydd


## 6. Save the subset for your own work

Write the text and the OCR scores to a local Parquet file. Keep this for your own
noncommercial use — do not redistribute the text (see the licence note at the top).

In [15]:
subset = yiddish_text.merge(
    yiddish[["barcode_src", "date1_src", "ocr_score_src"]],
    on="barcode_src", how="left",
)
subset.to_parquet("institutional-books-yiddish.parquet")
print("saved institutional-books-yiddish.parquet")

saved institutional-books-yiddish.parquet


## Recap

That is the whole pattern, and it is reusable for any language or topic:

1. **Filter small.** One ~306 MB metadata file (no token) answers "which books?" — pull
   it once, then every query is local and instant.
2. **Estimate.** A rough MB-per-book figure tells you whether the subset is safe to take
   whole *before* you spend any bandwidth.
3. **Fetch narrow.** A `WHERE barcode_src IN (...)` sweep pulls only the books you
   selected, in MB not GB.
4. **Keep the scores.** Saving `ocr_score_src` alongside the text tells you which books
   are ready to read and which need care.

## Other common selections

The language filter above is just one shape of the same idea: query the local metadata,
get back a set of `barcode_src`, then hand those to `fetch_text`. Here are three more
starting points. Each one returns barcodes you can pass straight to `fetch_text(...)`,
exactly as we did for Yiddish — and you can combine them (author *and* a date range,
subject *and* a language) by AND-ing the `WHERE` clauses.

You can create your own DuckDB queries from plain text using many LLMs.

In [ ]:
# By author. Names in catalogue records vary in spelling and transliteration, so match
# loosely with ILIKE (case-insensitive substring). Sholem Aleichem was catalogued both
# under his pen name and his birth name, Rabinowitz.
by_author = con.sql(f"""
    SELECT barcode_src, title_src, author_src, date1_src, language_gen
    FROM '{META}'
    WHERE author_src ILIKE '%sholem%aleichem%'
       OR author_src ILIKE '%rabinowitz%'
    ORDER BY TRY_CAST(date1_src AS INTEGER)
""").df()

print(f"{len(by_author)} books")
by_author.head(10)
# by_author_text = fetch_text(by_author.barcode_src.tolist())

In [ ]:
# By genre / subject. Library of Congress subject headings live in `topic_or_subject_src`
# (comma-joined, and often null). ILIKE matches any heading that contains the term.
by_subject = con.sql(f"""
    SELECT barcode_src, title_src, topic_or_subject_src, date1_src, language_gen
    FROM '{META}'
    WHERE topic_or_subject_src ILIKE '%folklore%'
    ORDER BY TRY_CAST(date1_src AS INTEGER)
""").df()

print(f"{len(by_subject):,} books")
by_subject.head(10)
# by_subject_text = fetch_text(by_subject.barcode_src.tolist())

In [ ]:
# By date range. `date1_src` is a messy MARC string ("18uu", "", ranges). TRY_CAST turns
# anything non-numeric into NULL, which quietly drops out of the BETWEEN instead of erroring.
by_year = con.sql(f"""
    SELECT barcode_src, title_src, author_src, date1_src, language_gen
    FROM '{META}'
    WHERE TRY_CAST(date1_src AS INTEGER) BETWEEN 1900 AND 1910
    ORDER BY TRY_CAST(date1_src AS INTEGER)
""").df()

print(f"{len(by_year):,} books published 1900–1910")
by_year.head(10)
# by_year_text = fetch_text(by_year.barcode_src.tolist())